# Seoul bike-sharing demand: hourly rental forecast

**Capstone 1, Machine Learning Zoomcamp 2026.** This notebook holds the data preparation and cleaning, the EDA and feature importance, and the model selection and hyper-parameter tuning. Its last code cell writes `params.json`, which `train.py` reads to train the final model.

**Problem.** Predict how many public bikes will be rented in Seoul in a given hour, from the calendar (date, hour, public holiday) and the weather forecast for that hour. An operator can use the forecast to decide how many bikes to stock at stations and when to schedule rebalancing and maintenance.

**Data.** UCI *Seoul Bike Sharing Demand* (id 560, CC BY 4.0): one row per hour from 1 December 2017 to 30 November 2018. Download it with `python download_data.py`.

**How the evaluation is set up**
- *Chronological hold-out.* The last 20 % of the calendar days form the test set and stay untouched until section 9. Everything before (EDA, model selection, tuning) uses the training period only.
- *Blocked cross-validation.* Neighbouring hours have almost the same weather and demand, so a random split would leak information. Cross-validation therefore keeps whole 7-day blocks together (`GroupKFold`).
- *Metrics.* RMSE (used to select models), MAE, R2 and WAPE (total absolute error divided by total demand), all in bikes per hour except R2 and WAPE.

**To run it:** activate the project's Python 3.11 environment, `cd` into the project folder and execute all cells (or `jupyter nbconvert --to notebook --execute --inplace notebook.ipynb`).

| Section | |
|---|---|
| 1 | Load the raw data |
| 2 | Data quality checks |
| 3 | Cleaning |
| 4 | Chronological split |
| 5 | Exploratory data analysis (training period) |
| 6 | Baselines and cross-validation |
| 7 | Hyper-parameter tuning |
| 8 | Choosing the model |
| 9 | Evaluation on the test period (once) |
| 10 | Feature importance |
| 11 | Why the validation scheme matters |
| 12 | Writing `params.json` |

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display
from scipy import stats
from scipy.stats import loguniform, randint, uniform
from sklearn.feature_selection import mutual_info_regression
from sklearn.inspection import permutation_importance
from sklearn.model_selection import GridSearchCV, GroupKFold, KFold, RandomizedSearchCV

if not Path("common.py").exists():
    raise RuntimeError("Open this notebook from the project folder (the one that contains common.py).")
import common  # feature engineering, pipelines and helpers shared with train.py and predict.py

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 150)
sns.set_theme(style="whitegrid")
plt.rcParams["figure.dpi"] = 110

SEED = common.RANDOM_STATE
TARGET = common.TARGET
DATA_PATH = Path("data") / common.CSV_NAME

## 1. Load the raw data

`common.load_raw` reads the file as latin-1 (the degree sign is a single byte), maps the headers to clean names, parses the `dd/mm/yyyy` dates and turns the categorical columns into booleans or standard labels. It stops with a clear message if a column is missing or a category is unexpected, so nothing downstream is built on a guessed schema.

| Column | Original header | Meaning | Unit |
|---|---|---|---|
| `date` | Date | calendar day (stored as day/month/year) | |
| `rented_bike_count` | Rented Bike Count | **target**: bikes rented in that hour | bikes per hour |
| `hour` | Hour | hour of the day | 0 to 23 |
| `temperature_c` | Temperature(°C) | air temperature | °C |
| `humidity_pct` | Humidity(%) | relative humidity | % |
| `wind_speed_ms` | Wind speed (m/s) | wind speed | m/s |
| `visibility_10m` | Visibility (10m) | visibility | units of 10 m |
| `dew_point_c` | Dew point temperature(°C) | dew point | °C |
| `solar_radiation_mj_m2` | Solar Radiation (MJ/m2) | solar radiation | MJ/m² |
| `rainfall_mm` | Rainfall(mm) | rainfall in the hour | mm |
| `snowfall_cm` | Snowfall (cm) | snowfall in the hour | cm |
| `season` | Seasons | Winter, Spring, Summer, Autumn | |
| `holiday` | Holiday | public holiday (`True`) or not | |
| `functioning_day` | Functioning Day | the bike system was operating in that hour (`True`) or not | |

In [ ]:
if not DATA_PATH.exists():
    raise FileNotFoundError("data/SeoulBikeData.csv not found. Run `python download_data.py` from the project folder first.")

raw = common.load_raw(DATA_PATH)
print(f"{DATA_PATH.name}: {len(raw)} rows x {raw.shape[1]} columns")
print(f"First day: {raw['date'].min().date()}    last day: {raw['date'].max().date()}")
display(raw.head())
display(raw.dtypes.astype(str).to_frame("dtype").T)

## 2. Data quality checks

Missing values, duplicates, completeness of the hourly series, value ranges, categories and a few physical plausibility checks.

In [ ]:
missing = raw.isna().sum()
print("Missing values per column (non-numeric cells would also show up here):")
print(missing.to_string())
print()
print("Exact duplicate rows:", int(raw.duplicated().sum()))
print("Duplicate (date, hour) pairs:", int(raw.duplicated(subset=["date", "hour"]).sum()))

hours_per_day = raw.groupby("date").size()
span_days = (raw["date"].max() - raw["date"].min()).days + 1
print(f"Calendar days present: {len(hours_per_day)} of {span_days} between the first and the last date")
print("Number of days by hours recorded per day:", hours_per_day.value_counts().to_dict())

In [ ]:
print("Value ranges of the numeric columns:")
display(raw[common.NUMERIC_RAW_COLUMNS].describe().T.round(2))
for column in ["season", "holiday", "functioning_day"]:
    print(f"\n{column}:")
    print(raw[column].value_counts().to_string())

In [ ]:
closed = raw[~raw["functioning_day"]]
open_hours = raw[raw["functioning_day"]]
print(f"Hours with the bike system switched off: {len(closed)} ({len(closed) / len(raw):.1%} of all hours)")
print("Rentals recorded in those hours: maximum", closed[TARGET].max(), "| all zero:", bool((closed[TARGET] == 0).all()))
print("Operating hours with zero rentals:", int((open_hours[TARGET] == 0).sum()))
closed_per_day = closed.groupby("date").size()
print(f"Days with closed hours: {len(closed_per_day)} "
      f"(whole days: {int((closed_per_day == 24).sum())}, partial days: {int((closed_per_day < 24).sum())})")

print()
zero_humidity = raw[raw["humidity_pct"] == 0]
print("Hours with a relative humidity of exactly 0 %:", len(zero_humidity))
if len(zero_humidity):
    display(zero_humidity[["date", "hour", "temperature_c", "humidity_pct", "dew_point_c"]].head(8))
print("Hours with a dew point above the air temperature:", int((raw["dew_point_c"] > raw["temperature_c"]).sum()))
print(f"Visibility at its maximum value of {raw['visibility_10m'].max():.0f}: "
      f"{(raw['visibility_10m'] == raw['visibility_10m'].max()).mean():.1%} of hours")
print(f"Hours with rain: {(raw['rainfall_mm'] > 0).mean():.1%}    hours with snow: {(raw['snowfall_cm'] > 0).mean():.1%}")

meteorological = {12: "Winter", 1: "Winter", 2: "Winter", 3: "Spring", 4: "Spring", 5: "Spring",
                  6: "Summer", 7: "Summer", 8: "Summer", 9: "Autumn", 10: "Autumn", 11: "Autumn"}
agreement = (raw["date"].dt.month.map(meteorological) == raw["season"]).mean()
print(f"The file's season label matches the meteorological season of the date in {agreement:.1%} of hours")

### Cleaning rules

1. **Exact duplicate rows** are dropped (the check above counts them).
2. **Hours in which the system was switched off** are dropped. The rental count is 0 there by construction: the bikes could not be rented, so it is not demand. The model forecasts demand *given that the service operates*.
3. **A relative humidity of exactly 0 %** does not occur in Seoul, so it is treated as a missing reading. This rule lives in the feature-engineering step of the pipeline (not here), so that training and serving behave identically; the missing value is then replaced by the training median inside every cross-validation fold.
4. **Season** is not a model input: it is a coarse proxy for temperature and solar radiation, and the chronological hold-out contains almost all of the autumn days, so the model could not learn it. It is used for the EDA only.
5. Rows are sorted chronologically.

## 3. Cleaning

In [ ]:
data, cleaning_report = common.clean_with_report(raw)
display(pd.Series(cleaning_report, name="rows").to_frame())
print("Modelling table:", data.shape)
print("Missing values in the model inputs:", int(data[common.INPUT_COLUMNS].isna().sum().sum()))
display(data.head())

## 4. Chronological split

The test period is the last 20 % of the calendar days. From here on the notebook looks at the **training period only**.

In [ ]:
train_mask, test_mask, cutoff = common.time_split_masks(data["date"])
train = data.loc[train_mask].reset_index(drop=True)
test = data.loc[test_mask].reset_index(drop=True)

X_train, y_train = train[common.INPUT_COLUMNS], train[TARGET]
X_test, y_test = test[common.INPUT_COLUMNS], test[TARGET]
groups = common.block_groups(train["date"])   # 7-day blocks, used for blocked cross-validation

split_summary = pd.DataFrame(
    {
        "rows": [len(train), len(test)],
        "days": [train["date"].nunique(), test["date"].nunique()],
        "first day": [train["date"].min().date(), test["date"].min().date()],
        "last day": [train["date"].max().date(), test["date"].max().date()],
    },
    index=["training period", "test period (held out)"],
)
display(split_summary)
print(f"Cross-validation blocks in the training period: {len(np.unique(groups))} (each block = {common.BLOCK_DAYS} consecutive days)")

## 5. Exploratory data analysis (training period)

### 5.1 The target: bikes rented per hour

In [ ]:
target = train[TARGET]
display(target.describe().round(1).to_frame("rented bikes per hour").T)
print(f"Skewness: {stats.skew(target):.2f}    operating hours with zero rentals: {(target == 0).mean():.2%}")

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
sns.histplot(target, bins=50, ax=axes[0])
axes[0].set_title("Rentals per hour")
sns.histplot(np.log1p(target), bins=50, ax=axes[1])
axes[1].set_title("log(1 + rentals per hour)")
sns.boxplot(x=target, ax=axes[2])
axes[2].set_title("Boxplot of rentals per hour")
plt.tight_layout()
plt.show()

### 5.2 Demand over time

In [ ]:
daily_mean = train.groupby("date")[TARGET].mean()
fig, ax = plt.subplots(figsize=(13, 3.8))
ax.plot(daily_mean.index, daily_mean.values, lw=1)
ax.set_ylabel("mean rentals per operating hour")
ax.set_title("Daily average demand, training period (the test period starts after the last day shown)")
plt.tight_layout()
plt.show()

monthly = (
    train.assign(month=train["date"].dt.to_period("M").astype(str))
    .groupby("month")[TARGET]
    .agg(hours="size", mean="mean", median="median", maximum="max")
    .round(0)
)
display(monthly)

### 5.3 Hour of the day, working days versus days off

In [ ]:
day_off = common.build_features(X_train)["day_off"].to_numpy() == 1
eda = train.assign(day_type=np.where(day_off, "day off (weekend or holiday)", "working day"))

profile = (
    eda.groupby(["day_type", "hour"])[TARGET]
    .agg(mean="mean", q25=lambda s: s.quantile(0.25), q75=lambda s: s.quantile(0.75))
    .reset_index()
)
fig, ax = plt.subplots(figsize=(11, 4.2))
for label, part in profile.groupby("day_type"):
    ax.plot(part["hour"], part["mean"], marker="o", label=label)
    ax.fill_between(part["hour"], part["q25"], part["q75"], alpha=0.15)
ax.set_xticks(range(0, 24, 2))
ax.set_xlabel("hour of the day")
ax.set_ylabel("rented bikes per hour")
ax.set_title("Average demand by hour (band: 25th to 75th percentile)")
ax.legend()
plt.tight_layout()
plt.show()

for label, part in profile.groupby("day_type"):
    series = part.set_index("hour")["mean"]
    print(f"{label}: busiest hour {int(series.idxmax()):02d}:00 ({series.max():.0f} bikes/h), "
          f"quietest hour {int(series.idxmin()):02d}:00 ({series.min():.0f} bikes/h)")

### 5.4 Weekday and hour

In [ ]:
weekday_names = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]
heat = (
    train.assign(weekday=train["date"].dt.dayofweek)
    .pivot_table(index="weekday", columns="hour", values=TARGET, aggfunc="mean")
)
heat.index = [weekday_names[i] for i in heat.index]
fig, ax = plt.subplots(figsize=(13, 3.8))
sns.heatmap(heat, cmap="viridis", ax=ax, cbar_kws={"label": "mean rentals per hour"})
ax.set_xlabel("hour of the day")
ax.set_ylabel("")
ax.set_title("Average demand by weekday and hour")
plt.tight_layout()
plt.show()

weekdays_only = train[train["date"].dt.dayofweek < 5]
holiday_effect = weekdays_only.groupby("holiday")[TARGET].agg(hours="size", mean="mean", median="median").round(1)
holiday_effect.index = [("public holiday on a weekday" if flag else "regular weekday") for flag in holiday_effect.index]
print("Public holidays that fall on a weekday versus regular weekdays:")
display(holiday_effect)

### 5.5 Seasons and months

The season label is used here for exploration only (see the cleaning rules). The training period contains only the first days of autumn, because the autumn days are mostly in the hold-out test period.

In [ ]:
season_order = ["Winter", "Spring", "Summer", "Autumn"]
fig, axes = plt.subplots(1, 2, figsize=(14, 4.2))
sns.boxplot(data=train, x="season", y=TARGET, order=season_order, ax=axes[0])
axes[0].set_title("Rentals per hour by season")
axes[0].set_xlabel("")
month_frame = train.assign(month=train["date"].dt.month)
sns.boxplot(data=month_frame, x="month", y=TARGET, ax=axes[1])
axes[1].set_title("Rentals per hour by month")
plt.tight_layout()
plt.show()
print("Hours per season in the training period:")
print(train["season"].value_counts().reindex(season_order).fillna(0).astype(int).to_string())

### 5.6 Weather

In [ ]:
temperature_bins = pd.cut(train["temperature_c"], bins=list(range(-20, 45, 5)))
by_temperature = train.groupby(temperature_bins, observed=True)[TARGET].agg(hours="size", mean="mean", median="median").round(0)
by_temperature.index = by_temperature.index.astype(str)
display(by_temperature)

fig, axes = plt.subplots(2, 3, figsize=(16, 8))
for ax, column in zip(axes.ravel(), ["temperature_c", "humidity_pct", "wind_speed_ms", "visibility_10m", "solar_radiation_mj_m2", "dew_point_c"]):
    ax.scatter(train[column], train[TARGET], s=4, alpha=0.2)
    quantile_bins = pd.qcut(train[column], 20, duplicates="drop")
    means = train.groupby(quantile_bins, observed=True).agg(x=(column, "mean"), y=(TARGET, "mean"))
    ax.plot(means["x"], means["y"], color="red", lw=2)
    ax.set_xlabel(column)
    ax.set_ylabel("rentals per hour")
fig.suptitle("Demand against each weather variable (red: mean in 20 equally populated bins)", y=1.0)
plt.tight_layout()
plt.show()

In [ ]:
precipitation = np.select(
    [train["snowfall_cm"] > 0, train["rainfall_mm"] > 0], ["snow", "rain"], default="dry"
)
frame = train.assign(precipitation=precipitation)
print("All operating hours:")
display(frame.groupby("precipitation")[TARGET].agg(hours="size", mean="mean", median="median").round(0))
rush = frame[(frame["hour"].isin([8, 18])) & (frame["date"].dt.dayofweek < 5)]
print("Weekday commuting hours only (08:00 and 18:00), which removes most of the time-of-day effect:")
display(rush.groupby("precipitation")[TARGET].agg(hours="size", mean="mean", median="median").round(0))

### 5.7 Correlations and model-free feature relevance

Spearman correlation only captures monotonic relationships, so it understates the effect of the hour of the day (low at night, high at the commuting peaks). Mutual information, computed on the engineered features, also detects non-monotonic and interaction-free dependence on the target.

In [ ]:
numeric_columns = [TARGET, "hour"] + common.WEATHER_COLUMNS
corr = train[numeric_columns].corr(method="spearman")
fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0, ax=ax)
ax.set_title("Spearman correlation (training period)")
plt.tight_layout()
plt.show()

with_target = corr[TARGET].drop(TARGET).sort_values(key=abs, ascending=False).round(2)
display(with_target.to_frame("Spearman correlation with rentals"))
print(f"Correlation between temperature and dew point: {corr.loc['temperature_c', 'dew_point_c']:.2f} "
      "(strongly related inputs: the regularised linear model copes with this; trees are not affected)")

In [ ]:
features = common.build_features(X_train).drop(columns=["hour_daytype"])
features = features.fillna(features.median())
discrete = [column in ("hour", "day_of_week", *common.BINARY_COLUMNS) for column in features.columns]
mutual_information = pd.Series(
    mutual_info_regression(features, y_train, discrete_features=discrete, random_state=SEED),
    index=features.columns,
).sort_values()

fig, ax = plt.subplots(figsize=(8, 6))
mutual_information.plot.barh(ax=ax)
ax.set_xlabel("mutual information with rentals per hour (nats)")
ax.set_title("Model-free feature relevance (training period)")
plt.tight_layout()
plt.show()

### 5.8 What the EDA means for the model

The plots and tables above motivate the choices implemented in `common.py`:

- **Hour and day type interact.** The hourly profile differs between working days and days off, so the features include the day of the week, a public-holiday flag, a combined `day_off` flag and a combined `hour_daytype` category (used by the linear model).
- **Precipitation is zero-inflated.** Most hours are dry, so rain and snow get on/off flags, and the linear model applies `log1p` to the amounts.
- **The target is right-skewed**, so the linear model can be fitted on `log(1 + count)`. Section 7 compares this with the raw count instead of assuming it.
- **Temperature and dew point carry overlapping information**, which a ridge penalty handles.
- **Season is left out** of the model for the reasons given in the cleaning rules.

## 6. Baselines and cross-validation

Two naive predictors set the bar, then the four candidate models are scored with their default hyper-parameters:

- the overall training mean (a constant),
- the average demand of the same hour and day type (working day or day off): what a planner would do without a weather forecast,
- a regularised linear model (ridge), a decision tree, a random forest and gradient-boosted trees (`HistGradientBoostingRegressor`).

All scores come from blocked 5-fold cross-validation on the training period.

In [ ]:
MODELS = common.MODEL_NAMES


def reference_cv(predict_fn):
    """Blocked CV of a naive predictor ``predict_fn(fit_part, valid_part) -> predictions``."""
    folds = []
    for fit_idx, valid_idx in GroupKFold(n_splits=common.N_SPLITS).split(X_train, y_train, groups):
        prediction = predict_fn(train.iloc[fit_idx], train.iloc[valid_idx])
        folds.append(common.regression_metrics(y_train.iloc[valid_idx], prediction))
    return common.summarise_folds(folds)


def summary_table(summaries):
    table = pd.DataFrame(summaries).T[["rmse_mean", "rmse_std", "mae_mean", "r2_mean", "wape_mean"]]
    table.columns = ["CV RMSE", "RMSE std", "CV MAE", "CV R2", "CV WAPE"]
    return table.sort_values("CV RMSE")


default_cv = {
    "training mean (constant)": reference_cv(lambda fit, valid: np.full(len(valid), fit[TARGET].mean())),
    "average of same hour and day type": reference_cv(common.profile_baseline),
}
for name in MODELS:
    default_cv[name] = common.cross_validate_model(common.build_pipeline(name), X_train, y_train, groups)["summary"]

baseline_table = summary_table(default_cv)
display(baseline_table.round(3))

fig, ax = plt.subplots(figsize=(9, 4))
ax.barh(baseline_table.index[::-1], baseline_table["CV RMSE"].to_numpy()[::-1], xerr=baseline_table["RMSE std"].to_numpy()[::-1])
ax.set_xlabel("cross-validated RMSE (bikes per hour, lower is better)")
ax.set_title("Baselines and default models")
plt.tight_layout()
plt.show()

## 7. Hyper-parameter tuning

- **Ridge:** a grid over the regularisation strength `alpha`, run twice, once with the target `log(1 + count)` and once with the raw count. The notebook keeps the variant with the lower cross-validated RMSE.
- **Decision tree, random forest, gradient boosting:** `RandomizedSearchCV` with a fixed budget of random configurations.

Every search uses the same blocked folds and the same scorer as the final evaluation (RMSE of predictions clipped at 0, exactly what the service returns). This can take several minutes.

In [ ]:
N_ITER = {"decision_tree": 30, "random_forest": 15, "hist_gradient_boosting": 30}
RIDGE_ALPHAS = np.logspace(-2, 3, 11)

SEARCH_SPACES = {
    "decision_tree": {
        "model__max_depth": randint(3, 26),
        "model__min_samples_leaf": randint(2, 61),
    },
    "random_forest": {
        "model__n_estimators": randint(150, 401),
        "model__max_depth": randint(8, 31),
        "model__min_samples_leaf": randint(2, 11),
        "model__max_features": uniform(0.3, 0.7),          # fraction of the features, 0.3 to 1.0
    },
    "hist_gradient_boosting": {
        "model__learning_rate": loguniform(0.02, 0.3),
        "model__max_iter": randint(150, 701),
        "model__max_leaf_nodes": randint(8, 64),
        "model__min_samples_leaf": randint(5, 81),
        "model__l2_regularization": loguniform(1e-3, 10),
    },
}
folds = GroupKFold(n_splits=common.N_SPLITS)

In [ ]:
ridge_rows = []
for log_target in (True, False):
    grid = GridSearchCV(
        common.build_pipeline("ridge", {"log_target": log_target}),
        param_grid={"model__regressor__alpha": RIDGE_ALPHAS},
        scoring=common.RMSE_SCORER,
        cv=folds,
        n_jobs=-1,
    )
    grid.fit(X_train, y_train, groups=groups)
    ridge_rows.append(
        {
            "target": "log(1 + count)" if log_target else "raw count",
            "log_target": log_target,
            "best alpha": float(grid.best_params_["model__regressor__alpha"]),
            "CV RMSE": float(-grid.best_score_),
        }
    )
ridge_table = pd.DataFrame(ridge_rows)
display(ridge_table.round(3))
best_ridge = min(ridge_rows, key=lambda row: row["CV RMSE"])
print(f"Ridge: keep the {best_ridge['target']} target with alpha = {best_ridge['best alpha']:.3g}")

In [ ]:
searches = {}
for name in ("decision_tree", "random_forest", "hist_gradient_boosting"):
    search = RandomizedSearchCV(
        common.build_pipeline(name),
        param_distributions=SEARCH_SPACES[name],
        n_iter=N_ITER[name],
        scoring=common.RMSE_SCORER,
        cv=folds,
        random_state=SEED,
        n_jobs=-1,
    )
    search.fit(X_train, y_train, groups=groups)
    searches[name] = search
    print(f"{name}: best CV RMSE {-search.best_score_:.1f}   {common.plain_params(search.best_params_)}")

In [ ]:
for name, search in searches.items():
    results = pd.DataFrame(search.cv_results_)
    results["CV RMSE"] = -results["mean_test_score"]
    results["RMSE std"] = results["std_test_score"]
    print(f"{name}: the five best of {len(results)} configurations")
    display(results.sort_values("CV RMSE").head(5)[["params", "CV RMSE", "RMSE std"]].round(2))

## 8. Choosing the model

The searched hyper-parameters are scored once more with the same helper that `train.py` uses, so that the numbers here and in `reports/metrics.json` are comparable. For every model the notebook keeps the better of the starting values (from `common.py`) and the searched values, so tuning can never make a model worse. The model with the lowest cross-validated RMSE is then selected: the test period plays no role in this choice.

In [ ]:
searched_params = {
    "ridge": {"alpha": best_ridge["best alpha"], "log_target": best_ridge["log_target"]},
    **{name: common.plain_params(search.best_params_) for name, search in searches.items()},
}
searched_cv = {
    name: common.cross_validate_model(common.build_pipeline(name, searched_params[name]), X_train, y_train, groups)["summary"]
    for name in MODELS
}

final_params, final_cv, kept = {}, {}, {}
for name in MODELS:
    if searched_cv[name]["rmse_mean"] <= default_cv[name]["rmse_mean"]:
        final_params[name], final_cv[name], kept[name] = searched_params[name], searched_cv[name], "searched"
    else:
        final_params[name], final_cv[name], kept[name] = dict(common.DEFAULT_PARAMS[name]), default_cv[name], "starting values"

comparison = pd.DataFrame(
    {
        "starting-values CV RMSE": {name: default_cv[name]["rmse_mean"] for name in MODELS},
        "searched CV RMSE": {name: searched_cv[name]["rmse_mean"] for name in MODELS},
        "kept": kept,
        "CV RMSE (kept)": {name: final_cv[name]["rmse_mean"] for name in MODELS},
        "CV MAE (kept)": {name: final_cv[name]["mae_mean"] for name in MODELS},
        "CV R2 (kept)": {name: final_cv[name]["r2_mean"] for name in MODELS},
    }
)
display(comparison.sort_values("CV RMSE (kept)").round(3))

best_name = comparison["CV RMSE (kept)"].astype(float).idxmin()
print(f"Selected model: {best_name}")
print("Hyper-parameters:", final_params[best_name])

fig, ax = plt.subplots(figsize=(9, 4))
comparison[["starting-values CV RMSE", "searched CV RMSE"]].plot.bar(ax=ax, rot=0)
ax.set_ylabel("cross-validated RMSE (bikes per hour)")
ax.set_title("Starting values versus searched hyper-parameters")
plt.tight_layout()
plt.show()

## 9. Evaluation on the test period (once)

The selected pipeline is fitted on the training period and scored on the held-out test period, next to the two naive predictors. These are the numbers to report. (`train.py` finally refits the same pipeline on **all** days for deployment; that model has seen the test period and cannot be scored on it.)

In [ ]:
best_pipeline = common.build_pipeline(best_name, final_params[best_name]).fit(X_train, y_train)
test_prediction = common.predict_counts(best_pipeline, X_test)

test_scores = pd.DataFrame(
    {
        f"{best_name} (selected)": common.regression_metrics(y_test, test_prediction),
        "average of same hour and day type": common.regression_metrics(y_test, common.profile_baseline(train, test)),
        "training mean (constant)": common.regression_metrics(y_test, np.full(len(test), y_train.mean())),
    }
).T
display(test_scores.round(3))

selected_scores = test_scores.iloc[0]
print(f"On the {len(test)} hours of the test period the selected model is off by {selected_scores['mae']:.0f} bikes per hour on average "
      f"(RMSE {selected_scores['rmse']:.0f}, R2 {selected_scores['r2']:.3f}); "
      f"its absolute error is {selected_scores['wape']:.1%} of the total demand.")

In [ ]:
residual = y_test.to_numpy() - test_prediction

fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
axes[0].scatter(y_test, test_prediction, s=5, alpha=0.3)
limit = float(max(y_test.max(), test_prediction.max()))
axes[0].plot([0, limit], [0, limit], color="red", lw=1)
axes[0].set_xlabel("actual rentals per hour")
axes[0].set_ylabel("predicted rentals per hour")
axes[0].set_title("Predicted versus actual (test period)")
sns.histplot(residual, bins=50, ax=axes[1])
axes[1].set_xlabel("actual minus predicted")
axes[1].set_title("Residuals")
sns.boxplot(x=test["hour"], y=residual, ax=axes[2])
axes[2].axhline(0, color="red", lw=1)
axes[2].set_xlabel("hour of the day")
axes[2].set_ylabel("actual minus predicted")
axes[2].set_title("Residuals by hour")
plt.tight_layout()
plt.show()

stamps = test["date"] + pd.to_timedelta(test["hour"], unit="h")
actual_series = pd.Series(y_test.to_numpy(), index=stamps).groupby(level=0).mean().asfreq("h")
predicted_series = pd.Series(test_prediction, index=stamps).groupby(level=0).mean().asfreq("h")
window = actual_series.index >= actual_series.index.max() - pd.Timedelta(days=14)
fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(actual_series[window], label="actual", lw=1.2)
ax.plot(predicted_series[window], label="predicted", lw=1.2)
ax.set_ylabel("rentals per hour")
ax.set_title("Last 14 days of the test period (gaps: hours with the system switched off)")
ax.legend()
fig.autofmt_xdate()
plt.tight_layout()
plt.show()

In [ ]:
def error_breakdown(frame, predictions, labels):
    """MAE, bias and WAPE of the predictions inside each group of ``labels``."""
    out = frame[[TARGET]].copy()
    out["prediction"] = predictions
    out["error"] = out["prediction"] - out[TARGET]
    out["abs_error"] = out["error"].abs()
    grouped = out.groupby(np.asarray(labels), observed=True)
    table = pd.DataFrame(
        {
            "hours": grouped.size(),
            "mean actual": grouped[TARGET].mean(),
            "MAE": grouped["abs_error"].mean(),
            "bias (pred - actual)": grouped["error"].mean(),
        }
    )
    table["WAPE %"] = 100 * grouped["abs_error"].sum() / grouped[TARGET].sum()
    return table.round(1)


test_features = common.build_features(X_test)
groups_for_breakdown = {
    "day type": np.where(test_features["day_off"] == 1, "day off", "working day"),
    "time of day": pd.cut(test["hour"], [-1, 5, 9, 15, 19, 23], labels=["00-05", "06-09", "10-15", "16-19", "20-23"]),
    "precipitation": np.select([test["snowfall_cm"] > 0, test["rainfall_mm"] > 0], ["snow", "rain"], default="dry"),
}
for title, labels in groups_for_breakdown.items():
    print(f"Test-period error by {title}:")
    display(error_breakdown(test, test_prediction, labels))

## 10. Feature importance

Permutation importance on the test period: how much the RMSE grows when one **input** is shuffled while the others stay as they are. Correlated inputs (temperature, dew point, solar radiation) share their importance, so read the ranking as a guide and not as exact attribution.

In [ ]:
permutation = permutation_importance(
    best_pipeline, X_test, y_test, scoring=common.RMSE_SCORER, n_repeats=10, random_state=SEED, n_jobs=1
)
importance = pd.DataFrame(
    {"input": X_test.columns, "rmse_increase": permutation.importances_mean, "std": permutation.importances_std}
)
importance["input"] = importance["input"].replace({"date": "date (day of the week)"})
importance = importance.sort_values("rmse_increase")

fig, ax = plt.subplots(figsize=(9, 5))
ax.barh(importance["input"], importance["rmse_increase"], xerr=importance["std"])
ax.set_xlabel("increase of the test RMSE when the input is shuffled (bikes per hour)")
ax.set_title(f"Permutation importance ({best_name}, test period)")
plt.tight_layout()
plt.show()
display(importance.sort_values("rmse_increase", ascending=False).round(2).reset_index(drop=True))

In [ ]:
fitted_model = best_pipeline.named_steps["model"]
feature_names = best_pipeline.named_steps["preprocess"].get_feature_names_out()

if hasattr(fitted_model, "feature_importances_"):
    own = pd.Series(fitted_model.feature_importances_, index=feature_names).sort_values().tail(15)
    ax = own.plot.barh(figsize=(8, 5))
    ax.set_xlabel("impurity-based importance")
    ax.set_title(f"Model-based importance ({best_name}, top 15 of {len(feature_names)} model columns)")
    plt.tight_layout()
    plt.show()
elif best_name == "ridge":
    coefficients = pd.Series(fitted_model.regressor_.coef_, index=feature_names)
    top = coefficients.reindex(coefficients.abs().sort_values().tail(15).index)
    ax = top.plot.barh(figsize=(8, 5))
    ax.set_xlabel("coefficient")
    ax.set_title("Largest ridge coefficients")
    plt.tight_layout()
    plt.show()
else:
    print(f"{best_name} does not expose impurity-based importances; the permutation importance above is the guide.")

In [ ]:
# Partial dependence by hand: average prediction when one input is set to each value of a grid.
sample = X_train.sample(1500, random_state=SEED)
sample_off = common.build_features(sample)["day_off"].to_numpy() == 1


def dependence(frame, column, grid):
    values = []
    for value in grid:
        shifted = frame.copy()
        shifted[column] = value
        values.append(common.predict_counts(best_pipeline, shifted).mean())
    return np.array(values)


hours = np.arange(24)
temperatures = np.arange(-15, 37.5, 2.5)
fig, axes = plt.subplots(1, 2, figsize=(14, 4.2))
axes[0].plot(hours, dependence(sample[~sample_off], "hour", hours), marker="o", label="working days")
axes[0].plot(hours, dependence(sample[sample_off], "hour", hours), marker="o", label="days off")
axes[0].set_xlabel("hour of the day")
axes[0].set_ylabel("average predicted rentals per hour")
axes[0].set_title("Partial dependence on the hour")
axes[0].legend()
axes[1].plot(temperatures, dependence(sample, "temperature_c", temperatures), marker="o")
axes[1].set_xlabel("temperature (°C)")
axes[1].set_ylabel("average predicted rentals per hour")
axes[1].set_title("Partial dependence on the temperature")
plt.tight_layout()
plt.show()

The curves keep all other inputs at their observed values, so unusual combinations (for example 03:00 together with the sunshine of an afternoon) enter the average. Treat them as a sanity check of the learned shape, not as exact causal effects.

## 11. Why the validation scheme matters

The same selected pipeline is cross-validated on the training period twice: with shuffled rows (neighbouring hours of the same day end up on both sides of the split) and with the 7-day blocks used everywhere else. Only the training period is used here; the test period stays untouched.

In [ ]:
selected_pipeline = common.build_pipeline(best_name, final_params[best_name])
shuffled = common.cross_validate_model(
    selected_pipeline, X_train, y_train, None, splitter=KFold(n_splits=common.N_SPLITS, shuffle=True, random_state=SEED)
)["summary"]
blocked = final_cv[best_name]
leak_table = pd.DataFrame(
    {"blocked 7-day folds (used)": blocked, "shuffled rows (leaky)": shuffled}
).loc[["rmse_mean", "mae_mean", "r2_mean", "wape_mean"]]
leak_table.index = ["RMSE", "MAE", "R2", "WAPE"]
display(leak_table.round(3))

gap = blocked["rmse_mean"] - shuffled["rmse_mean"]
if gap > 0:
    print(f"Shuffled rows report an RMSE that is {gap:.1f} bikes per hour ({gap / blocked['rmse_mean']:.1%}) lower than blocked folds: "
          "optimism that comes from the model having seen the same day's neighbouring hours.")
else:
    print("On this data shuffled rows do not look better than blocked folds (RMSE difference "
          f"{gap:.1f} bikes per hour); blocked folds are still the safer estimate for forecasting the future.")

## 12. Writing `params.json`

The tuned hyper-parameters are saved for `train.py`. The file holds no timestamps, so running the notebook again with the same data gives the same file.

In [ ]:
meta = {
    "status": "tuned",
    "method": "GridSearchCV for ridge (log and raw target), RandomizedSearchCV for the tree models; the better of the "
              "searched and the starting values is kept; scoring: RMSE of predictions clipped at 0; "
              "GroupKFold(5) over 7-day blocks of the training period",
    "n_iter": N_ITER,
    "kept": kept,
    "seed": SEED,
}
common.save_params("params.json", final_params, meta)
print(Path("params.json").read_text(encoding="utf-8"))

## Summary

In [ ]:
print(f"Selected model: {best_name}")
print(f"Cross-validated RMSE on the training period: {final_cv[best_name]['rmse_mean']:.1f} bikes per hour")
print(f"Test period: RMSE {selected_scores['rmse']:.1f}, MAE {selected_scores['mae']:.1f}, R2 {selected_scores['r2']:.3f}, WAPE {selected_scores['wape']:.1%}")
print("Next: run `python train.py` to train the final model and write reports/metrics.json.")